# EPUB Audiobook - batch chunk synthesis (multiple patches, one run)

This notebook synthesizes the text chunks exported by the EPUB Audiobook App
for **every patch in the batch**, sequentially, using the same `VoxCPM2` model
the app uses locally. For each patch it writes `chunk_NNN.wav` files into an
`output/` subfolder inside that patch's folder, and as soon as a patch is
complete it merges the chunks into a single **`result/NNN - <patch name>.wav`**
at the batch root.

> **Enable a GPU first.** This model runs on CUDA. In Colab: **Runtime > Change runtime type > GPU (T4)**, then restart the session. Pick **GPU, not TPU** - VoxCPM cannot use a TPU and will silently fall back to CPU (extremely slow). Cell 6 checks this for you.

It reads everything it needs from `batch_manifest.json`, which was exported
alongside this notebook - you should not need to type any patch info by hand.

## No re-downloading the model on restart
Cell 1 points the Hugging Face cache at persistent storage, so the multi-GB
`VoxCPM2` weights are downloaded **once**, not on every session:

- **Colab**: cached in your Drive at `EPUB Audiobook Exports/.cache` (the
  first run is a normal download that lands in Drive; later sessions load
  from there). Uses a few GB of Drive space.
- **Kaggle**: cached in `/kaggle/working/.cache` - turn on
  **Persistence: Files only** in the notebook options so it survives new
  sessions.

## Disconnects are fine - just re-run
Free Colab/Kaggle sessions can die mid-run. The synthesis cell is safe to
re-run any number of times:

- chunks that already have a `.wav` are **skipped**,
- patches that already have their merged `result/` file are **skipped entirely**,
- on Colab every `.wav` is written **directly into your Drive folder**, so
  progress survives even if the runtime is killed - reconnect, run the cells
  top to bottom (the model reloads), and it continues where it stopped.

## Google Colab (recommended)
The app already uploaded this folder into **your own Google Drive** (the
account you connected). Just run the cells top to bottom: cell 3 mounts your
Drive, and the folder is a normal filesystem path from then on - no Google API
calls needed in this notebook at all. Merged patch files end up in the
`result/` subfolder of the batch folder in Drive.

## Kaggle (recommended: Drive via Kaggle Secret)
Kaggle has no native Google Drive mount, but Cell 4 talks to the Drive API
directly using the app's own credentials, giving Kaggle the same experience
as Colab. One-time setup:

1. In the app, open the **Google Drive** page and click
   **Copy Kaggle credentials** (requires Drive to be connected).
2. On Kaggle: **Add-ons > Secrets** > add a secret named **`GDRIVE_CREDS`**
   with that JSON as the value, and enable it for this notebook.

Cell 4 then downloads the exported batch folder from Drive into
`/kaggle/working` (including any `.wav` chunks from earlier sessions), and
Cell 8 uploads every generated `.wav` and merged `result/` file **straight
back to Drive** as it is written. A dead Kaggle session resumes exactly like
Colab - re-run the cells and it continues - and the app can import the
results from Drive as usual.

### Kaggle without Drive (zip-dataset fallback)
If you'd rather not store credentials on Kaggle:
1. In the app, use **Download selected (.zip)**, upload the zip as a Kaggle
   Dataset and attach it to this notebook.
2. Skip Cells 3 and 4 and set `FOLDER_PATH` to the dataset path
   (e.g. `/kaggle/input/<dataset-name>`) - see the comment in Cell 4.
3. Output goes under `/kaggle/working`; run the last cell to zip `result/`
   and download it from Kaggle's **Output** pane.
4. **Resuming across sessions:** `/kaggle/working` does not survive a new
   session. To resume, add the `.wav` files you already produced to the
   dataset under `patches/patch_NNN/output/` - the skip check looks there
   too and will not redo those chunks.

## Video rendering (Cell 10–11)
After TTS synthesis is complete, **Cell 10** checks FFmpeg is available
(pre-installed on Colab/Kaggle). **Cell 11** then renders one MP4 per patch:
- Uses the background image bundled in the package (already has book title +
  patch name baked in by the app — no font needed here).
- Mixes the TTS audio with optional background music from `music/` (loop,
  low volume), if the book had music assigned when exported.
- Saves each MP4 to `result/NNN - <patch name>.mp4`.
- Skip-safe: if the MP4 already exists it is not re-rendered.

## YouTube upload (Cell 12)
**Cell 12** uploads every rendered MP4 directly to YouTube using a
`YOUTUBE_CREDS` Kaggle/Colab Secret. To set it up:
1. In the app, open **/youtube** and click **Copy YouTube credentials for Kaggle/Colab**.
2. On Kaggle: **Add-ons > Secrets** > add secret `YOUTUBE_CREDS` with that JSON.
   On Colab: left sidebar key icon > add secret `YOUTUBE_CREDS`.
- Skip-safe: if a `result/NNN.mp4.youtube_id` file exists, that video is skipped.

In [ ]:
# Cell 1: persistent caches + dependencies. Points the Hugging Face model cache
# (and pip's download cache) at persistent storage, so restarting the session
# does NOT re-download the multi-GB model weights every time:
#   - Colab: cached in your Google Drive under "EPUB Audiobook Exports/.cache".
#     The first run downloads the model once into Drive; every later session
#     loads it from there.
#   - Kaggle: cached in /kaggle/working/.cache. Enable the notebook's
#     "Persistence: Files only" setting (right sidebar > Notebook options) so
#     the cache survives across sessions - without it the cache still helps
#     within one session, but a brand-new session starts empty.
import os

USE_PERSISTENT_CACHE = True  # set False to use the default ephemeral cache

CACHE_ROOT = None
if USE_PERSISTENT_CACHE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')  # also mounted by Cell 3; mounting twice is fine
        CACHE_ROOT = "/content/drive/MyDrive/EPUB Audiobook Exports/.cache"
    except ImportError:
        if os.path.isdir("/kaggle"):
            CACHE_ROOT = "/kaggle/working/.cache"

if CACHE_ROOT:
    os.makedirs(CACHE_ROOT, exist_ok=True)
    # Must be set BEFORE anything imports huggingface_hub (it reads HF_HOME at
    # import time), which is why this cell comes first.
    os.environ["HF_HOME"] = os.path.join(CACHE_ROOT, "huggingface")
    # Drive's FUSE mount doesn't support symlinks; the HF cache detects that and
    # falls back to plain file copies - silence the warning about it.
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
    os.environ["PIP_CACHE_DIR"] = os.path.join(CACHE_ROOT, "pip")
    print("Persistent cache:", CACHE_ROOT)
else:
    print("No persistent cache - the model will be re-downloaded each session.")

!pip install -q voxcpm soundfile numpy

In [ ]:
# Cell 2: (optional) Hugging Face token - avoids the "unauthenticated requests" rate
# limit warning/slow downloads when fetching the model. Get a free token at
# https://huggingface.co/settings/tokens
#
# Recommended: store it as a secret instead of pasting it in plain text here -
# Colab: left sidebar > key icon > add secret named HF_TOKEN.
# Kaggle: Add-ons > Secrets > add secret named HF_TOKEN.
# If no secret is found, you'll get a hidden prompt to paste it manually (or just
# press Enter to skip and continue unauthenticated).
# The app replaces __HF_TOKEN__ below with the token from its own settings on export;
# leave the app's HF_TOKEN setting empty to keep this as a placeholder (secrets/prompt).
HF_TOKEN = "__HF_TOKEN__"

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    pass

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("Hugging Face token (leave blank to skip): ")

if HF_TOKEN:
    import os
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF token set - continuing unauthenticated (may hit rate limits).")

In [ ]:
# Cell 3: Google Colab only - mount your Drive. The exported batch folder is located
# automatically by batch id, so you do NOT need to paste the folder name by hand.
# Skip this cell entirely on Kaggle (use Cell 4 instead).
from google.colab import drive
import glob, json, os

drive.mount('/content/drive')

BATCH_ID = "__BATCH_ID__"  # injected by the app when this notebook was exported
EXPORTS_ROOT = "/content/drive/MyDrive/EPUB Audiobook Exports"
DEFAULT_FOLDER = os.path.join(EXPORTS_ROOT, "__DEFAULT_FOLDER_NAME__")

# Scan every export folder and match the one whose batch_manifest.json has our batch id.
FOLDER_PATH = None
for d in sorted(glob.glob(os.path.join(EXPORTS_ROOT, "*")), reverse=True):
    manifest_path = os.path.join(d, "batch_manifest.json")
    if not os.path.isfile(manifest_path):
        continue
    try:
        with open(manifest_path, encoding="utf-8") as f:
            if json.load(f).get("batch_id") == BATCH_ID:
                FOLDER_PATH = d
                break
    except Exception:
        pass

if FOLDER_PATH is None:
    FOLDER_PATH = DEFAULT_FOLDER  # fall back to the exact name the app used at export time

print("Using folder:", FOLDER_PATH)
assert os.path.isdir(FOLDER_PATH), (
    f"Folder not found: {FOLDER_PATH}\n"
    "Make sure the export was uploaded to this Google account's Drive, "
    "or set FOLDER_PATH manually."
)

In [ ]:
# Cell 4: Kaggle only - connect to Google Drive with the GDRIVE_CREDS secret and
# download the exported batch folder. Skip this cell on Colab.
#
# One-time setup (see the intro above): in the app open the Google Drive page and
# click "Copy Kaggle credentials", then on Kaggle add it via Add-ons > Secrets as a
# secret named GDRIVE_CREDS and enable it for this notebook.
#
# This cell downloads the batch (including output/ wavs from earlier sessions) into
# /kaggle/working/batch, and defines drive_persist() which Cell 8 uses to upload
# every generated .wav and merged result file straight back to Drive.
#
# --- zip-dataset fallback (no Drive credentials) ---
# Skip this cell too and point FOLDER_PATH at the attached dataset instead. Use the
# EXACT zip filename (without .zip) as the dataset name when uploading, e.g.:
# FOLDER_PATH = "/kaggle/input/<dataset-name>"
!pip install -q google-api-python-client google-auth

import io
import json
import os

from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from kaggle_secrets import UserSecretsClient

BATCH_ID = "__BATCH_ID__"  # injected by the app when this notebook was exported

creds_info = json.loads(UserSecretsClient().get_secret("GDRIVE_CREDS"))
creds = Credentials(
    token=None,
    refresh_token=creds_info["refresh_token"],
    token_uri="https://oauth2.googleapis.com/token",
    client_id=creds_info["client_id"],
    client_secret=creds_info["client_secret"],
    scopes=["https://www.googleapis.com/auth/drive.file"],
)
drive_service = build("drive", "v3", credentials=creds)

FOLDER_MIME = "application/vnd.google-apps.folder"


def _list_children(folder_id):
    files, token = [], None
    while True:
        resp = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=token, pageSize=1000,
        ).execute()
        files += resp.get("files", [])
        token = resp.get("nextPageToken")
        if not token:
            return files


def _download(file_id, dest):
    request = drive_service.files().get_media(fileId=file_id)
    downloader = MediaIoBaseDownload(dest, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()


# Locate the batch folder: scan "EPUB Audiobook Exports" for the folder whose
# batch_manifest.json carries this notebook's batch id.
resp = drive_service.files().list(
    q=f"name = 'EPUB Audiobook Exports' and mimeType = '{FOLDER_MIME}' and trashed = false",
    fields="files(id)",
).execute()
_roots = resp.get("files", [])
assert _roots, (
    "No 'EPUB Audiobook Exports' folder found. The credentials only see files "
    "created by the app itself - make sure you exported this batch to Drive."
)

_batch_folder_id = None
for _root in _roots:
    for _folder in _list_children(_root["id"]):
        if _folder["mimeType"] != FOLDER_MIME:
            continue
        _mf = next((f for f in _list_children(_folder["id"]) if f["name"] == "batch_manifest.json"), None)
        if _mf is None:
            continue
        _buf = io.BytesIO()
        _download(_mf["id"], _buf)
        if json.loads(_buf.getvalue().decode("utf-8")).get("batch_id") == BATCH_ID:
            _batch_folder_id = _folder["id"]
            print("Found batch folder on Drive:", _folder["name"])
            break
    if _batch_folder_id:
        break
assert _batch_folder_id, f"No Drive folder found with batch_id {BATCH_ID}"

# Download the whole batch folder - including output/ wavs from earlier sessions,
# which the skip and merge logic in Cell 8 needs locally - and remember the Drive
# ids of every folder/file so drive_persist() can upload without re-listing.
FOLDER_PATH = "/kaggle/working/batch"
_drive_folder_ids = {"": _batch_folder_id}
_drive_file_ids = {}


def _sync_down(folder_id, rel):
    for f in _list_children(folder_id):
        child_rel = f"{rel}/{f['name']}" if rel else f["name"]
        if f["mimeType"] == FOLDER_MIME:
            _drive_folder_ids[child_rel] = f["id"]
            _sync_down(f["id"], child_rel)
        else:
            _drive_file_ids[child_rel] = f["id"]
            local = os.path.join(FOLDER_PATH, child_rel)
            if not os.path.exists(local):
                os.makedirs(os.path.dirname(local), exist_ok=True)
                with open(local, "wb") as fh:
                    _download(f["id"], fh)
                print("downloaded", child_rel)


_sync_down(_batch_folder_id, "")
print("Batch ready at", FOLDER_PATH)


def drive_persist(local_path, rel_dir):
    """Upload a freshly written file into rel_dir inside the batch folder on Drive,
    creating the subfolder chain as needed and replacing an existing file in place.
    Cell 8 calls this after every chunk .wav and every merged result file, so a dead
    Kaggle session loses nothing."""
    parent, rel = _drive_folder_ids[""], ""
    for part in [p for p in rel_dir.split("/") if p]:
        rel = f"{rel}/{part}" if rel else part
        if rel not in _drive_folder_ids:
            folder = drive_service.files().create(
                body={"name": part, "mimeType": FOLDER_MIME, "parents": [parent]},
                fields="id",
            ).execute()
            _drive_folder_ids[rel] = folder["id"]
        parent = _drive_folder_ids[rel]
    name = os.path.basename(local_path)
    file_rel = f"{rel}/{name}" if rel else name
    media = MediaFileUpload(local_path)
    if file_rel in _drive_file_ids:
        drive_service.files().update(fileId=_drive_file_ids[file_rel], media_body=media).execute()
    else:
        created = drive_service.files().create(
            body={"name": name, "parents": [parent]}, media_body=media, fields="id",
        ).execute()
        _drive_file_ids[file_rel] = created["id"]

In [ ]:
# Cell 5: load the batch manifest and the shared voice reference clip (if the book
# uses voice cloning - the clip is book-level, so all patches share it)
import json
import os

with open(os.path.join(FOLDER_PATH, "batch_manifest.json"), "r", encoding="utf-8") as f:
    batch_manifest = json.load(f)

print(f"Batch of {batch_manifest['patch_count']} patches from book '{batch_manifest['book_title']}'")
for entry in batch_manifest["patches"]:
    print(f"  patch {entry['patch_index']:03d}: {entry['patch_name']} "
          f"(chapters {entry['chapter_start']}-{entry['chapter_end']}, {entry['chunk_count']} chunks)")

reference_wav_path = None
prompt_text = None
if batch_manifest.get("reference_wav"):
    reference_wav_path = os.path.join(FOLDER_PATH, batch_manifest["reference_wav"])
    prompt_text = batch_manifest.get("reference_transcript") or None
    print(f"Using cloned voice reference: {reference_wav_path}")

In [ ]:
# Cell 6: check you're actually on a GPU (VoxCPM is far too slow on CPU).
# If this stops with an error, go to Runtime > Change runtime type > Hardware
# accelerator > GPU (T4), then Runtime > Restart session and run again.
# NOTE: pick GPU, NOT TPU - VoxCPM uses CUDA and cannot run on a TPU.
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected - VoxCPM would run on CPU and be extremely slow. "
        "Colab: Runtime > Change runtime type > GPU (T4), then Restart session. "
        "Choose GPU, not TPU. On Kaggle: enable a GPU accelerator in the sidebar."
    )

In [ ]:
# Cell 7: load the model ONCE - it is reused for every patch in the batch.
# Kept in its own cell so that after a disconnect you can re-run Cell 8 alone
# without waiting for a re-download.
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained(batch_manifest.get("voxcpm_model_id", "openbmb/VoxCPM2"), load_denoiser=False)

In [ ]:
# Cell 8: synthesize every patch in order, merging each patch into result/ as soon
# as it completes. Safe to re-run after any disconnect:
#   - chunks with an existing .wav are skipped,
#   - patches with an existing merged result file are skipped entirely,
#   - on Colab everything below is written straight into the Drive-mounted folder,
#     so progress is persisted the moment each file is written.
import json
import os

import numpy as np
import soundfile as sf

# --- config ---
PATCH_IDS = None      # None = run ALL patches in the batch; or e.g. [12, 15] to restrict
SKIP_EXISTING = True  # skip chunks/patches that already have output (safe resume)

# Drive upload hook: defined by the Kaggle Drive cell (Cell 4); a no-op everywhere
# else - on Colab files are already written straight into the Drive mount, and the
# zip-dataset fallback has nowhere to upload to.
persist = globals().get("drive_persist") or (lambda local_path, rel_dir: None)

# /kaggle/input is a read-only mount, so with the zip-dataset fallback all output
# goes under /kaggle/working instead. Otherwise FOLDER_PATH is writable (the Drive
# mount on Colab, or /kaggle/working/batch in Kaggle Drive mode), so output/ and
# result/ live right inside the batch folder.
ON_KAGGLE_DATASET = FOLDER_PATH.startswith("/kaggle/input")
WORK_ROOT = "/kaggle/working" if ON_KAGGLE_DATASET else FOLDER_PATH
RESULT_DIR = os.path.join(WORK_ROOT, "result")
os.makedirs(RESULT_DIR, exist_ok=True)
print("Merged patch files will be written to:", RESULT_DIR)

sample_rate = model.tts_model.sample_rate
summary = []

for entry in sorted(batch_manifest["patches"], key=lambda e: e["patch_index"]):
    label = f"patch {entry['patch_index']:03d} ({entry['patch_name']})"
    if PATCH_IDS is not None and entry["patch_id"] not in PATCH_IDS:
        summary.append((label, "skipped (not in PATCH_IDS)"))
        continue

    patch_dir = os.path.join(FOLDER_PATH, entry["folder"])
    out_dir = os.path.join(WORK_ROOT, entry["folder"], "output")
    result_path = os.path.join(WORK_ROOT, entry["result_wav"])
    print(f"\n=== {label}: {entry['chunk_count']} chunks ===")

    if SKIP_EXISTING and os.path.exists(result_path):
        print(f"already merged -> {result_path} (skipping patch)")
        summary.append((label, "done (already merged)"))
        continue

    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(patch_dir, "manifest.json"), "r", encoding="utf-8") as f:
        manifest = json.load(f)

    def find_wav(wav_name):
        # Look in this run's output dir first, then in the exported folder itself -
        # on Colab both are the same Drive path; on Kaggle the second one catches
        # .wav files uploaded back into the read-only dataset for cross-session resume.
        for candidate in (os.path.join(out_dir, wav_name),
                          os.path.join(patch_dir, "output", wav_name)):
            if os.path.exists(candidate):
                return candidate
        return None

    for chunk_filename in manifest["chunks"]:
        index = chunk_filename.split("_")[1].split(".")[0]  # chunk_000.txt -> 000
        wav_name = f"chunk_{index}.wav"
        if SKIP_EXISTING and find_wav(wav_name):
            print(f"skip {chunk_filename} (already synthesized)")
            continue

        with open(os.path.join(patch_dir, chunk_filename), "r", encoding="utf-8") as f:
            text = f.read()

        kwargs = {}
        if reference_wav_path:
            kwargs["reference_wav_path"] = reference_wav_path
            if prompt_text:
                kwargs["prompt_wav_path"] = reference_wav_path
                kwargs["prompt_text"] = prompt_text

        audio = model.generate(text=text, cfg_value=2.0, inference_timesteps=10, **kwargs)
        out_path = os.path.join(out_dir, wav_name)
        sf.write(out_path, audio, sample_rate)
        persist(out_path, entry["folder"] + "/output")
        print(f"wrote {out_path}")

    # Merge this patch right away (instead of one big merge at the end) so an
    # interrupted batch still yields finished result files for completed patches.
    missing = [w for w in manifest["expected_outputs"] if find_wav(w) is None]
    if missing:
        print(f"patch incomplete - {len(missing)} chunk(s) missing "
              f"(first: {missing[0]}); re-run this cell to resume")
        summary.append((label, f"incomplete ({len(missing)} chunks missing)"))
        continue

    parts = []
    merge_sr = None
    for wav_name in manifest["expected_outputs"]:
        audio, sr = sf.read(find_wav(wav_name))
        if merge_sr is None:
            merge_sr = sr
        parts.append(audio)
    sf.write(result_path, np.concatenate(parts), merge_sr)
    persist(result_path, "result")
    print(f"merged {len(parts)} chunks -> {result_path}")
    summary.append((label, "merged"))

print("\n=== Batch summary ===")
for name, status in summary:
    print(f"- {name}: {status}")

In [ ]:
# Cell 9: Kaggle only - zip the merged result/ files so you can download them from
# the Output pane. In Kaggle Drive mode this is just a convenience copy (result/ is
# already uploaded to Drive); on Colab you don't need it at all.
import os
import shutil

if os.path.isdir("/kaggle"):
    archive = shutil.make_archive("/kaggle/working/results", "zip", RESULT_DIR)
    print("Download this from the Output pane:", archive)
else:
    print("Colab run - merged files are already saved in Drive:", RESULT_DIR)

In [ ]:
# Cell 10: Check FFmpeg is available (it is pre-installed on Colab/Kaggle Ubuntu).
# No font needed — background images already have text baked in by the app.
import subprocess

result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if result.returncode == 0:
    print("FFmpeg OK:", result.stdout.splitlines()[0])
else:
    raise RuntimeError("FFmpeg not found. Install with: !apt-get install -y ffmpeg")

In [ ]:
# Cell 11: Render MP4 per patch (resume-safe: skip if MP4 already exists).
# Supports optional background music and marquee (scrolling ticker bar).
import json
import os
import subprocess

video_config = batch_manifest.get("video_config", {})
music_rel = video_config.get("music_file")
music_volume = video_config.get("music_volume", 0.15)
fps = video_config.get("fps", 30)
resolution = video_config.get("resolution", "1920x1080")
w_h = resolution.replace("x", ":")

music_abs = os.path.join(FOLDER_PATH, music_rel) if music_rel else None
if music_abs and not os.path.exists(music_abs):
    print(f"Warning: music file not found at {music_abs} — rendering without music")
    music_abs = None

persist = globals().get("drive_persist") or (lambda local_path, rel_dir: None)

video_summary = []
for entry in sorted(batch_manifest["patches"], key=lambda e: e["patch_index"]):
    label = f"patch {entry['patch_index']:03d} ({entry['patch_name']})"
    result_mp4 = os.path.join(WORK_ROOT, entry["result_mp4"])

    if os.path.exists(result_mp4):
        print(f"skip {label} — MP4 already exists")
        video_summary.append((label, "skipped (exists)"))
        continue

    result_wav = os.path.join(WORK_ROOT, entry["result_wav"])
    if not os.path.exists(result_wav):
        print(f"skip {label} — WAV not ready (run TTS cells first)")
        video_summary.append((label, "skipped (no WAV)"))
        continue

    bg_rel = entry.get("background_image")
    bg_abs = os.path.join(FOLDER_PATH, bg_rel) if bg_rel else None
    if not bg_abs or not os.path.exists(bg_abs):
        print(f"skip {label} — background image not found")
        video_summary.append((label, "skipped (no background)"))
        continue

    # Marquee band: <patch_id>.marquee.png + <patch_id>.marquee.json bundled alongside background
    patch_dir = os.path.join(FOLDER_PATH, entry["folder"])
    patch_id = entry["patch_id"]
    marquee_png = os.path.join(patch_dir, f"{patch_id}.marquee.png")
    marquee_json = os.path.join(patch_dir, f"{patch_id}.marquee.json")
    marquee_meta = None
    if os.path.exists(marquee_png) and os.path.exists(marquee_json):
        with open(marquee_json, encoding="utf-8") as f:
            marquee_meta = json.load(f)

    print(f"\n=== Rendering {label} ===")
    os.makedirs(os.path.dirname(result_mp4), exist_ok=True)

    # Build inputs
    inputs = ["-loop", "1", "-i", bg_abs, "-i", result_wav]
    next_idx = 2
    music_idx = None
    marquee_idx = None
    if music_abs:
        inputs += ["-stream_loop", "-1", "-i", music_abs]
        music_idx = next_idx; next_idx += 1
    if marquee_meta:
        inputs += ["-loop", "1", "-i", marquee_png]
        marquee_idx = next_idx; next_idx += 1

    base_vf = f"scale={w_h}:force_original_aspect_ratio=decrease,pad={w_h}:(ow-iw)/2:(oh-ih)/2"
    audio_chains = []
    if music_idx is not None:
        audio_chains.append(f"[{music_idx}:a]volume={music_volume}[music]")
        audio_chains.append("[1:a][music]amix=inputs=2:duration=first:normalize=0[aout]")
        audio_map = "[aout]"
    else:
        audio_map = "1:a"

    if marquee_idx is not None and marquee_meta:
        band_h = marquee_meta["marquee_height"]
        speed = marquee_meta["speed_px_per_sec"]
        scroll_unit = max(1, marquee_meta["scroll_unit_px"])
        w = int(resolution.split("x")[0])
        band_vf = f"crop={w}:{band_h}:x='(t*{speed})%{scroll_unit}':y=0"
        chains = audio_chains + [
            f"[0:v]{base_vf}[bg]",
            f"[{marquee_idx}:v]{band_vf}[band]",
            "[bg][band]overlay=0:0[outv]",
        ]
        cmd = ["ffmpeg", "-y", *inputs,
               "-filter_complex", ";".join(chains),
               "-map", "[outv]", "-map", audio_map,
               "-c:v", "libx264", "-tune", "stillimage",
               "-r", str(fps), "-c:a", "aac", "-b:a", "192k",
               "-pix_fmt", "yuv420p", "-crf", "23", "-shortest", result_mp4]
    elif music_idx is not None:
        chains = audio_chains + [f"[0:v]{base_vf}"]
        cmd = ["ffmpeg", "-y", *inputs,
               "-filter_complex", ";".join(chains),
               "-map", "0:v", "-map", audio_map,
               "-c:v", "libx264", "-tune", "stillimage",
               "-r", str(fps), "-c:a", "aac", "-b:a", "192k",
               "-pix_fmt", "yuv420p", "-crf", "23", "-shortest", result_mp4]
    else:
        cmd = ["ffmpeg", "-y", *inputs,
               "-vf", base_vf,
               "-c:v", "libx264", "-tune", "stillimage",
               "-r", str(fps), "-c:a", "aac", "-b:a", "192k",
               "-pix_fmt", "yuv420p", "-crf", "23", "-shortest", result_mp4]

    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"FFmpeg error for {label}:")
        print(proc.stderr[-1000:])
        video_summary.append((label, "failed"))
        continue

    print(f"rendered -> {result_mp4}")
    persist(result_mp4, "result")
    video_summary.append((label, "done"))

print("\n=== Video render summary ===")
for name, status in video_summary:
    print(f"- {name}: {status}")

In [ ]:
# Cell 12: Upload rendered MP4s to YouTube (resume-safe: skip if .youtube_id exists).
# Reads YOUTUBE_CREDS secret (JSON with client_id, client_secret, refresh_token).
# Get it from the app: /youtube -> "Copy YouTube credentials for Kaggle/Colab".
import io
import json
import os

YOUTUBE_CREDS = None
try:
    from google.colab import userdata
    YOUTUBE_CREDS = json.loads(userdata.get("YOUTUBE_CREDS") or "{}")
except Exception:
    pass

if not YOUTUBE_CREDS:
    try:
        from kaggle_secrets import UserSecretsClient
        YOUTUBE_CREDS = json.loads(UserSecretsClient().get_secret("YOUTUBE_CREDS") or "{}")
    except Exception:
        pass

if not YOUTUBE_CREDS or not YOUTUBE_CREDS.get("refresh_token"):
    print("YOUTUBE_CREDS secret not found or empty.")
    print("To upload videos to YouTube from this notebook:")
    print("  1. In the app, open /youtube and click 'Copy YouTube credentials for Kaggle/Colab'")
    print("  2. On Kaggle: Add-ons > Secrets > add secret named YOUTUBE_CREDS with the JSON")
    print("  3. On Colab: left sidebar > key icon > add secret named YOUTUBE_CREDS")
    print("Skipping YouTube upload.")
else:
    from google.oauth2.credentials import Credentials
    from google.auth.transport.requests import Request
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload

    creds = Credentials(
        token=None,
        refresh_token=YOUTUBE_CREDS["refresh_token"],
        token_uri="https://oauth2.googleapis.com/token",
        client_id=YOUTUBE_CREDS["client_id"],
        client_secret=YOUTUBE_CREDS["client_secret"],
        scopes=["https://www.googleapis.com/auth/youtube.upload"],
    )
    creds.refresh(Request())
    youtube_service = build("youtube", "v3", credentials=creds)

    privacy = video_config.get("youtube_privacy", "private")
    upload_summary = []

    for entry in sorted(batch_manifest["patches"], key=lambda e: e["patch_index"]):
        label = f"patch {entry['patch_index']:03d} ({entry['patch_name']})"
        result_mp4 = os.path.join(WORK_ROOT, entry["result_mp4"])
        id_file = result_mp4 + ".youtube_id"

        if os.path.exists(id_file):
            vid_id = open(id_file).read().strip()
            print(f"skip {label} — already uploaded: https://youtube.com/watch?v={vid_id}")
            upload_summary.append((label, f"skipped (already uploaded: {vid_id})"))
            continue

        if not os.path.exists(result_mp4):
            print(f"skip {label} — MP4 not rendered yet (run Cell 11 first)")
            upload_summary.append((label, "skipped (no MP4)"))
            continue

        title = f"{batch_manifest['book_title']} - {entry['patch_name']}"
        print(f"\n=== Uploading {label} ===")
        print(f"  Title: {title}")

        request_body = {
            "snippet": {"title": title[:100], "description": f"{batch_manifest['book_title']} — audiobook", "categoryId": "26"},
            "status": {"privacyStatus": privacy},
        }
        media = MediaFileUpload(result_mp4, mimetype="video/mp4", resumable=True, chunksize=10 * 1024 * 1024)
        insert_req = youtube_service.videos().insert(part="snippet,status", body=request_body, media_body=media)

        response = None
        while response is None:
            status, response = insert_req.next_chunk()
            if status:
                print(f"  Upload progress: {int(status.progress() * 100)}%")

        video_id = response["id"]
        with open(id_file, "w") as f:
            f.write(video_id)
        persist(id_file, "result")
        print(f"  Uploaded: https://youtube.com/watch?v={video_id}")
        upload_summary.append((label, f"done: {video_id}"))

    print("\n=== YouTube upload summary ===")
    for name, status in upload_summary:
        print(f"- {name}: {status}")